# Inertial Measurement Unit (IMU): the basics

A Duckiedrone needs to keep itself level with the horizon to stay in the air. The sensor that
  makes this possible is the Inertial Measurement Unit, or IMU.

An **IMU** is a sensor package that measures **linear acceleration** along its own axes and
  **angular velocity** about them. It is a core component of every modern smart device, from
  phones to robots, and on the Duckiedrone it is built into the flight controller board.

The Duckiedrone puts this data to two different uses:

- **attitude**: the flight controller estimates how the Duckiedrone is tilted and corrects small
  perturbations, which is what keeps it horizontal when hovering.

- **motion**: integrating the measurements over time reconstructs velocity and position,
  although this second use comes with intrinsic limitations.

## Intended Learning Outcomes

By the end of the notebook you should be able to answer:

- What do the accelerometer and the gyroscope each measure, and in which reference frame?

- Why does a stationary accelerometer read 9.81 m/s^2 rather than zero?

- Why can gravity provide an absolute reference for roll and pitch, but not for yaw? (For a
  proper definition of roll, pitch, and yaw, see
  [Notebook 2 - Rotation Representations](./2-rotation-representations.ipynb).)

- What is the most significant advantage of IMUs?

- What is the most significant disadvantage of using IMUs?

- Why does integrating the accelerometer twice fail to give a usable position, and why does
  calibrating for longer not fix it?

## What does an IMU measure?

<figure style="width:420px; margin:1.5em auto; text-align:center;">
  <img src="../assets/_images/imu.png" style="width:100%;" />
  <figcaption style="font-size:0.9em; margin-top:0.6em; text-align:center;">
The two sensors inside an IMU: the accelerometer measures linear acceleration along the
  sensor's three axes (straight arrows), while the gyroscope measures angular velocity about them
  (curved arrows). <a
  href="https://api.semanticscholar.org/CorpusID:6415932">[source]</a>
  </figcaption>
</figure>

A typical IMU combines two sensors:

| Sensor | What does it measure | Physical Units |
|---|---|---|
| Accelerometer | Linear acceleration along sensor X / Y / Z | m/s^2 |
| Gyroscope     | Angular velocity about sensor X / Y / Z | rad/s |

Some IMUs add a magnetometer, which measures the Earth's magnetic field and so provides an
  absolute heading. The Duckiedrone's IMU does not include one.

### Notation

The symbols below are used throughout this learning experience. Bold marks a vector, and
  anything introduced later is defined where it first appears.

| Symbol | Meaning | Units |
|---|---|---|
| $a_x, a_y, a_z$ | linear acceleration of the sensor, along its own x, y, z axes | m/s^2 |
| $\tilde{a}_x, \tilde{a}_y, \tilde{a}_z$ | what the accelerometer reports for those axes | m/s^2 |
| $\dot{\theta}_x, \dot{\theta}_y, \dot{\theta}_z$ | angular velocity about the sensor's x, y, z axes | rad/s |
| $\mathbf{g}$ | gravitational acceleration, magnitude 9.81 m/s^2, pointing down | m/s^2 |
| $k$ | sample index, since measurements arrive at discrete instants $t_k$ | - |
| $\Delta t$ | time between two consecutive samples | s |
| $\mathbf{n}$ | the error term separating a measurement from the truth | m/s^2 |
| $b$ | the constant part of that error, the bias | m/s^2 |
| $\hat{r}$ | a stationary reference, estimated by averaging static samples | m/s^2 |

A tilde marks a measured quantity, as opposed to the true one it is meant to represent.

### The accelerometer

The principle of operation is a mass-spring-damper system: a small mass of known size on a
  spring, inside a box. Accelerate the box and the mass lags behind, stretching the spring. How
  far it stretches gives an instantaneous measurement of acceleration, provided the elastic
  constant of the spring is known.

<figure style="width:640px; margin:1.5em auto; text-align:center;">
  <img src="../assets/_images/mass_spring_damper.png" style="width:100%;" />
  <figcaption style="font-size:0.9em; margin-top:0.6em; text-align:center;">
An accelerometer as a mass-spring-damper. The box is fixed to the Duckiedrone, and the mass
  inside can only move by stretching the spring, so how far the spring stretches is what the
  sensor reads. At rest on a table the spring holds the weight of the mass and the reading is
  9.81 m/s^2 upward. In free fall the box and the mass fall together, the spring relaxes, and the
  reading is zero.
  </figcaption>
</figure>

Now leave the box sitting still on a table. What should the spring do?

It stays stretched. It has to hold the mass up against gravity, so a stationary accelerometer
  reads 9.81 m/s^2 pointing up rather than zero. Throw the box off the table and the opposite
  happens: mass and box fall together, the spring relaxes, and during the fall the reading drops
  to zero.

So the accelerometer does not report how fast the Duckiedrone is moving, nor its acceleration
  with respect to the ground. It reports acceleration relative to free fall, which has its own
  name: **proper acceleration**.

Writing $\mathbf{a}$ for the acceleration of the sensor with respect to the ground and
  $\mathbf{g}$ for gravitational acceleration, the measurement is

$$
\tilde{\mathbf{a}} = \mathbf{a} - \mathbf{g},
\qquad\text{component wise}\qquad
\tilde{a}_i = a_i - g_i, \quad i \in \{x, y, z\},
$$

with all three vectors resolved along the sensor's own axes. Gravity points down, towards the
  center of the Earth, with magnitude 9.81 m/s^2, so for a sensor lying level, with its z axis
  pointing up, $\mathbf{g} = (0, 0, -9.81)$ m/s^2.

Both cases above follow from that equation. On the table the sensor is not accelerating, so
  $\mathbf{a} = \mathbf{0}$ and $\tilde{\mathbf{a}} = -\mathbf{g} = (0, 0, +9.81)$ m/s^2,
  pointing **up**. In free fall gravity is the only acceleration acting, so
  $\mathbf{a} = \mathbf{g}$ and $\tilde{\mathbf{a}} = \mathbf{0}$.

Whether the sensor's axes line up with the Duckiedrone's own is a separate question, and the
  subject of [Notebook 2 - Rotation Representations](./2-rotation-representations.ipynb).

### The gyroscope

Attitude is what keeps the Duckiedrone level, and the gyroscope is what makes attitude available
  fast enough to act on. It measures the rate at which the sensor turns about each of its own
  axes,

$$
\dot{\theta}_i(t) = \frac{\mathrm{d}\theta_i}{\mathrm{d}t}, \quad i \in \{x, y, z\},
$$

in rad/s. Note that this is *not* the measurement of the heading (i.e., an angle), rather how
  fast the orientation is changing over time. Recovering the absolute orientation angle means
  integrating the rate,

$$
\theta_i(t) = \theta_i(0) + \int_0^t \dot{\theta}_i(\tau)\,\mathrm{d}\tau,
$$

which needs a starting angle $\theta_i(0)$ from somewhere else, and which accumulates whatever
  error the rate measurement carries.

Intuitively, imagine being spun around slowly in an office chair with closed eyes. How fast the
  chair is turning at any moment is relatively easy to feel. Saying which way it ended up facing
  after a while of spinning is not: that requires adding the rate up over the whole turn.

## Gathering IMU measurements from a Duckiedrone

Before going further into what the numbers mean, it is worth looking at them.

Open a shell on the Duckiedrone and read the topic the IMU data arrives on:

```bash
ssh duckie@<robot>.local
docker exec -it ros2-mavros bash
ros2 topic echo --once /mavros/imu/data
```

`<robot>` is the Duckiedrone's hostname. [Notebook 4](./4-interfacing-with-imu.ipynb) explains what
  each of these commands does and how to write a program that subscribes to this topic. Here the
  goal is only to look at the data. A virtual Duckiedrone publishes the same message, with values
  produced by the simulator instead of by a sensor; the Duckiematrix has to be running, with the
  Duckiedrone attached to an entity, or the topic stays silent.

The message below was captured on a physical Duckiedrone resting motionless on a bench:

```
header:
  stamp:
    sec: 1788950655
    nanosec: 961939565
  frame_id: base_link
orientation:
  x: 0.0035833670220291654
  y: 0.003122046427906428
  z: -0.7076347443857675
  w: -0.7065624650101054
orientation_covariance:
- 1.0
- 0.0
- 0.0
- 0.0
- 1.0
- 0.0
- 0.0
- 0.0
- 1.0
angular_velocity:
  x: 8.995339157991111e-06
  y: -0.0005893486086279153
  z: 0.000459553673863411
angular_velocity_covariance:
- 1.2184696791468346e-07
- 0.0
- 0.0
- 0.0
- 1.2184696791468346e-07
- 0.0
- 0.0
- 0.0
- 1.2184696791468346e-07
linear_acceleration:
  x: -0.018726181238889694
  y: -0.0861072316765773
  z: 9.91747760772705
linear_acceleration_covariance:
- 8.999999999999999e-08
- 0.0
- 0.0
- 0.0
- 8.999999999999999e-08
- 0.0
- 0.0
- 0.0
- 8.999999999999999e-08
---
```

This message is of the type `sensor_msgs/Imu`, and it carries three groups of numbers.

Two of them are the measurements defined above. `linear_acceleration` holds
  $(\tilde{a}_x, \tilde{a}_y, \tilde{a}_z)$ in m/s^2, and `angular_velocity` holds
  $(\dot{\theta}_x, \dot{\theta}_y, \dot{\theta}_z)$ in rad/s. The third, `orientation`, is not a
  measurement at all: no sensor in the package reports an angle. It is computed by the flight
  controller, and the next section explains how and why. Each group is followed by a covariance
  array, which describes how much the publisher believes its own numbers.

Now compare those readings against the theory. The Duckiedrone was still and level, so the previous
  section predicts $\tilde{\mathbf{a}} = -\mathbf{g} = (0, 0, +9.81)$ m/s^2 and
  $\dot{\boldsymbol{\theta}} = \mathbf{0}$.

The vertical reading is 9.917 m/s^2, not 9.81. The horizontal readings are -0.019 and -0.086
  m/s^2, not zero. The angular rates are small but not zero either. Nothing was moving, so the
  prediction was right and the numbers still do not match it.

That gap is not a mistake in the theory or a fault in the Duckiedrone. It is the subject of the
  rest of this notebook.

## Why the IMU data matters

The accelerometer and the gyroscope together are enough to work out which way the Duckiedrone is
  tilted, and that tilt is all the flight controller needs to stay level, as a horizontal drone
  has zero tilt. The same measurements could in principle also provide the position of the
  Duckiedrone through double integration, but in practice that estimate drifts away within
  seconds.

Neither sensor reports attitude directly. The flight controller derives it using an algorithm
  (an estimator called `EKF2`) that combines the calibrated accelerometer and gyroscope
  measurements into an educated guess ("estimate") of the Duckiedrone's orientation. That
  estimate is good for roll and pitch, but not for yaw.

Two words need separating before that sentence can mean anything.

An **absolute** measurement is one taken against a reference that exists outside the vehicle and
  does not move with it, so it can be trusted no matter how long the Duckiedrone has been
  flying. For example, having the GPS coordinates of the Duckiedrone would provide an absolute
  position measurement. A **relative** measurement only describes a change from wherever the
  vehicle happened to be, so its usefulness depends entirely on knowing that starting point and
  on nothing having gone wrong since. For example, saying the Duckiedrone has traveled ten
  meters in a certain direction, with respect to its takeoff location, is a relative
  measurement.

Gravity is an external reference, and this is why roll and pitch are recoverable. A stationary
  accelerometer reads 9.81 m/s^2 pointing directly opposite gravity, which always points towards
  the center of the Earth, the direction called down. Tilt the Duckiedrone and that reading
  redistributes across the three sensor axes in a way that depends on the tilt, so roll and
  pitch can be corrected against gravity rather than relying on accumulated gyroscope error.

Yaw has no such reference. Rotating the Duckiedrone about the vertical axis does not change the
  direction of gravity relative to the vehicle at all, so the accelerometer reading is identical
  at every heading and cannot tell one from another. A magnetometer would supply an outside
  reference by reading the Earth's magnetic field, but the Duckiedrone does not carry one.

Yaw therefore comes only from integrating the gyroscope, which is the office chair again. Any
  error in the measured rate is added into the total and keeps accumulating, so the heading
  estimate drifts slowly away from the truth.

## What an IMU does not measure

Given the above, an IMU alone cannot directly measure:

- **Position**: where the Duckiedrone is;

- **Linear velocity**: how fast it is moving;

- **Absolute yaw**: unless we add a magnetometer to the IMU.

Position, being the double integral of acceleration, is recoverable in theory. Once gravity has
  been subtracted and the measurement rotated into a world frame, which is what
  [Notebook 2](./2-rotation-representations.ipynb) is about, integrating twice should give it:

$$
\mathbf{v}(t) = \mathbf{v}(0) + \int_0^t \mathbf{a}(\tau)\,\mathrm{d}\tau,
\qquad
\mathbf{x}(t) = \mathbf{x}(0) + \int_0^t \mathbf{v}(\tau)\,\mathrm{d}\tau.
$$

Estimating position this way, by integrating a measurement of motion with no outside reference
  to correct it, is called **dead reckoning**.

Dead reckoning in general is not very useful, because the estimate tends to "quickly" diverge
  from the real value due to noise in the measurements. Every physical sensor is noisy, with no
  way around it: the electronics add a little noise of their own, the reading is rounded off
  when it is turned into a number, and the whole thing shifts as the sensor warms up. So a real
  measurement is the ideal one plus an error term:

$$
\tilde{\mathbf{a}}(t) = \mathbf{a}(t) - \mathbf{g} + \mathbf{n}(t).
$$

### Measuring the noise

The table below summarizes 1500 consecutive messages, 30 seconds at about 50 Hz, recorded from
  the same motionless, level Duckiedrone as the message above. A level, motionless sensor is
  expected to read $(0, 0, +9.81)$ m/s^2 and $(0, 0, 0)$ rad/s throughout.

| quantity | mean | standard deviation | minimum | maximum |
|---|---|---|---|---|
| $\tilde{a}_x$ [m/s^2] | -0.009 | 0.024 | -0.156 | +0.248 |
| $\tilde{a}_y$ [m/s^2] | -0.093 | 0.022 | -0.297 | +0.019 |
| $\tilde{a}_z$ [m/s^2] | +9.801 | 0.116 | +9.479 | +10.166 |
| $\dot{\theta}_x$ [rad/s] | +0.00002 | 0.00041 | -0.0016 | +0.0017 |
| $\dot{\theta}_y$ [rad/s] | +0.00001 | 0.00057 | -0.0036 | +0.0026 |
| $\dot{\theta}_z$ [rad/s] | +0.00005 | 0.00052 | -0.0022 | +0.0038 |

Two different effects are visible in that table:

The first is spread. Individual vertical readings range from 9.479 to 10.166 m/s^2 while the
  Duckiedrone does not move at all. The single message shown earlier happened to read 9.917.
  This part of $\mathbf{n}(t)$ is close to zero on average, so **averaging removes it**: the
  mean of 9.801 m/s^2 is within 0.006 of standard gravity, 9.80665 m/s^2.

The second is an offset that averaging does not remove. The mean of $\tilde{a}_y$ is -0.093
  m/s^2, and it stays there however many samples are added. Some of that is the bench not being
  perfectly level, and the rest belongs to the sensor. A roughly constant offset like this is
  called the **bias**, written $b$ for a single axis. It is a consequence of the sensor being a
  physical object, not an extra property that IMUs happen to have.

### Why the bias is fatal to dead reckoning

Now integrate a constant bias twice. A bias $b$ on one axis contributes a velocity error growing
  as $b\,t$ and a position error growing as

$$
\varepsilon(t) = \tfrac{1}{2}\,b\,t^{2},
$$

linear in time for velocity, quadratic for position. Nothing in the dead-reckoning calculation
  can tell that offset apart from genuine motion, because the two produce identical readings.

The Duckiedrone's IMU is an
  [InvenSense ICM-42688-P](https://invensense.tdk.com/products/motion-tracking/6-axis/icm-42688-p/).
  Datasheets quote accelerometer bias in milli-*g*, thousandths of gravity, so 1 mg is roughly
  0.01 m/s^2. The -0.093 m/s^2 measured on $\tilde{a}_y$ above is about 9 mg. A few mg is
  unremarkable for a sensor of this class, and it sounds harmless.

It is not harmless, and the quickest way to see why is to watch it happen.

### Activity 1: an ideal Duckiedrone with one flaw

This first demonstration is deliberately idealized, so that exactly one thing can go wrong. A
  Duckiedrone sits on the floor of a 2.5 m room, powered up and completely still. It never moves.
  Assume it already knows its orientation and has removed gravity perfectly, so the only defect
  left is a small constant bias $b$ on the vertical axis. It dead-reckons its altitude by
  integrating that reading twice, exactly as in the equation above.

Green is where the Duckiedrone really is. Red is where it believes it is.

Start at 5 mg and drag the time slider. Then try 1 mg, which would be an excellent sensor, and see
  whether it helps.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider

# Symbols follow the notation table above:
#   b     accelerometer bias on the vertical axis [m/s^2]
#   t_k   sample instants; a real IMU reports at discrete times, not continuously
#   x     dead-reckoned altitude [m]

G = 9.80665        # standard gravity, the magnitude of g [m/s^2]
SAMPLE_RATE = 50.0 # the Duckiedrone's IMU stream rate [Hz]
TRUE_ALT = 0.0     # the Duckiedrone sits on the floor and never moves [m]
CEILING = 2.5      # room height [m]


def ideal_drift(bias_mg=5.0, duration_s=60.0):
    """Compare where the Duckiedrone really is with where dead reckoning says it is."""
    b = bias_mg * G / 1000.0                       # milli-g -> m/s^2

    # The estimate exists only at the instants the sensor reports, so build those instants
    # explicitly rather than a continuous curve.
    t_k = np.arange(0.0, duration_s, 1.0 / SAMPLE_RATE)

    # Integrating a constant bias twice gives x = (1/2) b t^2, the closed form from the text.
    x = TRUE_ALT + 0.5 * b * t_k ** 2

    fig, ax = plt.subplots(figsize=(9, 5))

    ax.axhspan(CEILING, 10, color='0.92')          # everything above the ceiling
    ax.axhline(CEILING, color='0.5', lw=1)
    ax.text(duration_s * 0.99, CEILING + 0.06, 'ceiling',
            ha='right', fontsize=9, color='0.4')

    # Where it really is
    ax.axhline(TRUE_ALT, color='tab:green', lw=2.5, label='where it really is')

    # The estimate: markers, not a line, because nothing is known between samples. At 50 Hz the
    # markers would touch and look continuous, so only every `stride`-th one is drawn. The
    # estimate itself still uses every sample.
    stride = max(1, len(t_k) // 120)
    marker_gap = stride / SAMPLE_RATE
    ax.plot(t_k[::stride], x[::stride], color='tab:red', linestyle='none', marker='.',
            markersize=6,
            label=f'dead-reckoned altitude estimate (one dot per {marker_gap:.1f} s)')

    ax.set_ylim(-0.15, CEILING + 0.5)
    ax.set_yticks(np.arange(0.0, CEILING + 0.5, 0.5))
    ax.set_xlim(0, duration_s)
    ax.set_xlabel('time [s]')
    ax.set_ylabel('altitude [m]')
    ax.legend(loc='center right')

    ax.set_title(f'bias b = {bias_mg:.1f} mg = {b:.3f} m/s^2    '
                 f'after {t_k[-1]:.0f} s it is off by {x[-1] - TRUE_ALT:.1f} m')
    fig.tight_layout()
    plt.show()


interact(
    ideal_drift,
    bias_mg=FloatSlider(value=5.0, min=0.1, max=5.0, step=0.1,
                        description='bias [mg]', continuous_update=False),
    duration_s=FloatSlider(value=60.0, min=5.0, max=300.0, step=1.0,
                           description='time [s]', continuous_update=False),
);

The Duckiedrone never moved, yet within seconds its own estimate puts it above the ceiling.
  Reducing the bias delays that moment without changing the outcome, because the error grows
  with the square of time however small $b$ is.

A fair objection is that the plot above only demonstrates the formula
  $\varepsilon = \frac{1}{2}bt^2$, since the formula is what produced it. The next demonstration
  replaces the idealized calculation with a recording.

### Activity 2: the same experiment on a physical Duckiedrone

A physical Duckiedrone was placed on the floor of a room, powered up with its motors off, and
  left static for five minutes. Its accelerometer measurements were logged throughout at about
  50 Hz, and that log ships with this learning experience as
  [`assets/data/imu_still_5min.csv`](../assets/data/imu_still_5min.csv). These are recorded
  sensor measurements, not simulated ones. The red dots are the altitude obtained by double
  integrating every sample, though only every few samples is plotted so the dots stay
  distinguishable. The green line is where the Duckiedrone really is, which is known
  continuously: it never moved.

Measurements arrive at discrete instants rather than continuously, so the integrals above become
  sums over samples. Writing $\tilde{a}_z(k)$ for the vertical reading of sample $k$, and
  $\Delta t$ for the time between samples, the procedure is:

1. Hold the Duckiedrone still and collect $N$ samples over a calibration window
   $T_{calibration}$.

2. Average them into a stationary reference:

$$
\hat{r} = \frac{1}{N}\sum_{k=1}^{N} \tilde{a}_z(k).
$$

   This single number holds gravity and the bias together, and there is no way to separate the
     two from a static recording: both are constant offsets on a sensor that is not moving.

3. Subtract that reference from every later sample, and dead-reckon what is left:

$$
a_z(k) = \tilde{a}_z(k) - \hat{r}.
$$

   For a Duckiedrone that never moves, $a_z(k)$ should be zero, so every departure from zero is
     error.

The first slider sets $T_{calibration}$. The second sets how long the estimate then runs.
  Integration starts at the first sample after calibration ends, so `t = 0` on the plot is the
  start of integration, and the calculation begins from the known stationary state on the floor.

Averaging for longer gives a better estimate of $\hat{r}$, although it will never give a correct
  one, because the quantity being estimated does not hold still.

Try to find a calibration time that keeps the red dots inside the room, given a default ceiling
  of 2.5 m.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
from pathlib import Path

# Symbols follow the notation table above:
#   a_z_tilde   the measured vertical acceleration, so the tilde marks a measurement
#   t_k         the instant each sample was recorded [s]
#   r_hat       the stationary reference, averaged over the calibration window
#   a_z         what is left after subtracting that reference
#   v, x        the first and second integrals of a_z: velocity [m/s] and altitude [m]

CEILING = 2.5      # room height [m]
FLOOR = 0.0        # the Duckiedrone sits on the floor and never moves [m]

# One accelerometer axis from a Duckiedrone that sat still on the floor for five
# minutes, sampled at about 50 Hz. Columns: t [s], ax, ay, az [m/s^2].
data_path = Path('assets/data/imu_still_5min.csv')
if not data_path.exists():
    data_path = Path('../assets/data/imu_still_5min.csv')
data = np.genfromtxt(data_path, delimiter=',', names=True)
t_all, a_z_tilde_all = data['t'], data['az']


def real_drift(calibration_s=10.0, watch_s=60.0):
    """Dead reckon altitude from the recording, after calibrating for so many seconds."""
    # Step 2 of the procedure above: average the static samples. This reference holds gravity
    # and the bias together, which is why the whole of it is subtracted and not just the bias.
    r_hat = a_z_tilde_all[t_all < calibration_s].mean()

    # Dead reckoning only starts once calibration is over. At that moment the Duckiedrone knows
    # it is on the floor and not moving, so the clock and the estimate both start at zero.
    integrating = (t_all >= calibration_s) & (t_all <= calibration_s + watch_s)
    sample_times = t_all[integrating]
    t_k = sample_times - sample_times[0]

    # Step 3: what remains once the reference is removed. For a Duckiedrone that never moves
    # this should be zero, and every departure from zero is error.
    a_z = a_z_tilde_all[integrating] - r_hat

    # The integrals of the text become sums over samples, by the trapezoidal rule.
    dt = np.diff(t_k)
    v = np.concatenate(([0.0], np.cumsum(0.5 * (a_z[:-1] + a_z[1:]) * dt)))
    x = FLOOR + np.concatenate(([0.0], np.cumsum(0.5 * (v[:-1] + v[1:]) * dt)))

    fig, ax = plt.subplots(figsize=(9, 5))

    ax.axhspan(CEILING, 10, color='0.92')
    ax.axhspan(-10, FLOOR, color='0.88')
    ax.axhline(CEILING, color='0.5', lw=1)
    ax.text(t_k[-1] * 0.99, CEILING + 0.06, 'ceiling', ha='right', fontsize=9, color='0.4')

    # Where it really is
    ax.axhline(FLOOR, color='tab:green', lw=2.5, label='where it really is')

    # The estimate: These are measurements, and nothing is known about what
    # happened between two of them. At about 50 Hz the markers would touch and look continuous,
    # so only every `stride`-th one is drawn; the estimate itself uses every sample.
    stride = max(1, len(t_k) // 120)
    marker_gap = stride * float(np.median(dt))
    ax.plot(t_k[::stride], x[::stride], color='tab:red', linestyle='none', marker='.',
            markersize=6,
            label=f'dead-reckoned altitude estimate (one dot per {marker_gap:.1f} s)')

    ax.set_ylim(-1.0, CEILING + 0.5)
    ax.set_xlim(0, t_k[-1])
    ax.set_xlabel('time since integration started [s]')
    ax.set_ylabel('altitude [m]')
    ax.legend(loc='upper right')

    n_calib = len(t_all[t_all < calibration_s])
    ax.set_title(f'calibrated for {calibration_s:.0f} s over {n_calib} samples    '
                 f'after {t_k[-1]:.0f} s of dead reckoning it is off by {x[-1] - FLOOR:+.1f} m')
    fig.tight_layout()
    plt.show()


interact(
    real_drift,
    calibration_s=FloatSlider(value=10.0, min=1.0, max=120.0, step=1.0,
                              description='calibrate [s]', continuous_update=False),
    watch_s=FloatSlider(value=60.0, min=10.0, max=180.0, step=5.0,
                        description='watch [s]', continuous_update=False),
);

### What these two toy examples show

Ten seconds of calibration is already longer than anyone waits in practice, and the Duckiedrone
  still puts itself through the ceiling within twenty seconds of starting to integrate. Increasing
  the calibration time changes the drift without removing it: average two full minutes of a
  Duckiedrone that was sitting still, and over the three minutes of recording that remain the
  estimate ends up about 2.4 meters below the floor.

That second result is the one worth pausing on, because it shows the problem is not a bias that was
  measured carelessly. Averaged over the full five minutes, the stationary vertical reading is
  within a tenth of a milli-g of standard gravity, which is smaller than the smallest bias the
  first slider will accept. The overall average is excellent. What it hides is variation between
  short windows: the value measured during a ten-second calibration is already different from the
  value in the 60 to 70 second window, and double integration turns that small mismatch into
  meters.

So calibration delays the failure and never removes it. The culprit is not the sensor but the
  integration, which has no way to tell a small constant offset from real motion and no outside
  reference to check itself against.

## Summary and next steps

1. An IMU packages an accelerometer and a gyroscope, providing measurements of linear acceleration
   and angular velocity with respect to the sensor's own reference frame.

2. The accelerometer reports $\tilde{\mathbf{a}} = \mathbf{a} - \mathbf{g}$, which is why a
   stationary sensor reads 9.81 m/s^2 upward and a falling one reads zero. The gyroscope reports a
   rate, never an angle.

3. This data matters because it yields the Duckiedrone's attitude, which is what keeps the vehicle
   level. Gravity provides an absolute reference for roll and pitch; without a magnetometer, yaw
   has none.

4. IMU measurements are carried in a ROS 2 `sensor_msgs/Imu` message, alongside an orientation that
   the flight controller estimates rather than measures.

5. Position can be dead-reckoned by integrating twice, but every physical sensor has noise, a
   constant part of that noise is a bias, and double integrating a bias produces an error growing
   as $\frac{1}{2}bt^2$. Calibration delays this and cannot remove it, as the recording from a
   physical Duckiedrone showed.

This notebook has quietly assumed throughout that a measurement taken along the sensor's axes says
  something directly about the Duckiedrone as a whole. That is not generally true. It depends on
  the relative pose of three different reference frames: the sensor's, the vehicle's, and the
  world's. Relating them is what makes a measurement of the sensor into a statement about the
  Duckiedrone.

Reference frames, and how to move between them, are the subject of
  [Notebook 2](./2-rotation-representations.ipynb).